# Five-percent PHOENIX experiment

An alternative to notebooks 01 and 02. Run once with `MODE="extract"` on CPU, save the pose output as a Kaggle dataset, then run with `MODE="train"` on GPU and that dataset attached. This is a full experiment on 5% of training data, with full validation and test splits.


## 1. Configure the session


In [ ]:
MODE = "extract"  # extract or train
IMAGE_DIR = "/kaggle/input/datasets/thesisacc2/rwth-phoenix-2014-t"
POSE_DIR = "/kaggle/input/phoenix-poses"
OUTPUT_DIR = "/kaggle/working/poses"
GROUPS = "all"  # Use core for a smaller training experiment.
assert MODE in {"extract", "train"}


## 2. Install dependencies


In [ ]:
import subprocess
import sys

if MODE == "extract":
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                    "tensorflow", "tensorflow-cpu", "tf-keras", "keras",
                    "mediapipe", "mediapipe-nightly"], check=True)
    packages = ["mediapipe==0.10.14", "opencv-python-headless", "tqdm", "pandas"]
else:
    packages = ["sentencepiece", "sacrebleu", "pandas", "matplotlib", "tqdm"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


## 3. Load the repository


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/linhxm/Sign-Language-REL.git"
CODE_DIR = Path("/kaggle/working/Sign-Language-REL")
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True)
assert (CODE_DIR / "main.py").is_file(), f"Invalid repository: {CODE_DIR}"
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
os.environ["SLT_CODE_DIR"] = str(CODE_DIR)
print("Repository:", CODE_DIR)

import json

def run_project(script, *args, data=None, training=None):
    """Run a script with temporary settings in a fresh Python process."""
    overrides = json.dumps({"data": data or {}, "train": training or {}})
    bootstrap = """
import json, runpy, sys
from configs.config import CFG
for section, values in json.loads(sys.argv[1]).items():
    for key, value in values.items():
        setattr(getattr(CFG, section), key, value)
sys.argv = sys.argv[2:]
runpy.run_path(sys.argv[0], run_name="__main__")
"""
    subprocess.run(
        [sys.executable, "-c", bootstrap, overrides, script, *map(str, args)],
        cwd=CODE_DIR, check=True,
    )


## 4. Extract and export poses (CPU session)

Save `poses_5pct.zip` as a Kaggle dataset, then attach it in the training session. Existing pose files in `OUTPUT_DIR` are reused.


In [ ]:
if MODE == "extract":
    import shutil

    names_file = "/kaggle/working/subset_names.txt"
    run_project("scripts/make_subset_names.py", "--subset", "0.05", "--out", names_file)
    run_project("data/extract_poses.py", "--input_dir", IMAGE_DIR,
                "--out_dir", OUTPUT_DIR, "--names_file", names_file, "--workers", "0")
    archive = shutil.make_archive("/kaggle/working/poses_5pct", "zip", OUTPUT_DIR)
    print("Saved:", archive)


## 5. Train (GPU session)

Set `POSE_DIR` to the folder containing the extracted `.npz` files. Full experiment groups can take hours.


In [ ]:
if MODE == "train":
    assert list(Path(POSE_DIR).glob("*.npz")), f"No pose files in {POSE_DIR}"
    run_project("run_all.py", "--subset", "0.05", "--groups", GROUPS,
                data={"pose_cache_dir": POSE_DIR, "work_dir": "/kaggle/working"})


## 6. Review tables and figures


In [ ]:
# ---------- Xem kết quả: bảng render ĐẸP (pandas) + biểu đồ hiển thị INLINE ----------
if MODE == "train":
    import os, glob
    import pandas as pd
    from IPython.display import display, Image, Markdown

    WORK = "/kaggle/working"
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.max_colwidth", 40)

    def _show_csv(path, title, sort=None):
        if not os.path.exists(path):
            return
        df = pd.read_csv(path)
        if sort:
            df = df.sort_values([c for c in sort if c in df.columns], na_position="last")
        display(Markdown(f"### {title}"))
        display(df.reset_index(drop=True))   # DataFrame -> Kaggle render thành bảng HTML đẹp

    # 1) Bảng GỘP mọi run (thay cho việc print markdown thô -> giờ là bảng thật)
    _show_csv(os.path.join(WORK, "comparison_table.csv"),
              "Bảng gộp - tất cả run", sort=["subset_pct", "encoder", "method"])

    # 2) 6 bảng báo cáo đã lọc sẵn theo từng câu hỏi so sánh
    TB = os.path.join(WORK, "report", "tables")
    for name, title in [
        ("table_main",      "Exp 1/7 - XE vs mọi thuật toán RL (Transformer)"),
        ("table_encoders",  "Exp 4 - so sánh 6 encoder"),
        ("table_reward",    "Exp 9 - reward ablation (4 combo)"),
        ("table_ablations", "REINFORCE / A2C / Curriculum"),
        ("table_baseline",  "Baseline sàn (empty / most-frequent)"),
        ("table_latency",   "Exp 15 - latency / memory / params"),
    ]:
        _show_csv(os.path.join(TB, name + ".csv"), title)

    # 3) Các biểu đồ (PNG) hiển thị NGAY trong notebook
    FG = os.path.join(WORK, "report", "figures")
    pngs = sorted(glob.glob(os.path.join(FG, "*.png")))
    if pngs:
        display(Markdown("### Biểu đồ"))
        for p in pngs:
            display(Markdown(f"**{os.path.basename(p)}**"))
            display(Image(filename=p))
    else:
        print("Chưa có biểu đồ trong", FG, "- chạy xong run_all.py (nó tự gọi make_report) rồi mở lại.")
else:
    print("bỏ qua xem kết quả (MODE != 'train')")
